# Lab 3E: Linear Regression with Mixed Features

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "lab3e.csv"
TARGET_COL = "performance_score"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COL}")

display(df.head())

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print("Sample categorical columns:", categorical_cols[:10])

Dataset shape: (54600, 75)
Target column: performance_score


,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


Numeric columns: 61
Categorical columns: 14
Sample categorical columns: ['player_id', 'player_name', 'nationality', 'team', 'position', 'preferred_foot', 'club_name', 'match_id', 'match_date', 'stadium']


In [8]:
drop_cols = ["player_id", "player_name", "match_id", "match_date"]
feature_cols = [c for c in df.columns if c not in [TARGET_COL] + drop_cols]

X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Feature count: {len(feature_cols)}")
print(f"Numeric features used: {len(numeric_features)}")
print(f"Categorical features used: {len(categorical_features)}")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Missing values in X_train before preprocessing: {X_train.isna().sum().sum()}")

Feature count: 70
Numeric features used: 60
Categorical features used: 10
Train shape: (43680, 70), Test shape: (10920, 70)
Missing values in X_train before preprocessing: 0


In [9]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X_train, y_train)
print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [10]:
def get_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_metrics = get_metrics(y_train, train_pred)
test_metrics = get_metrics(y_test, test_pred)

results = pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"]).round(4)
print("Regression Metrics (Lower MAE/MSE/RMSE is better, Higher R2 is better):")
display(results)

Regression Metrics (Lower MAE/MSE/RMSE is better, Higher R2 is better):


,MAE,MSE,RMSE,R2
Train,1.9677,6.4989,2.5493,0.9933
Test,1.9756,6.4855,2.5467,0.9933


In [11]:
r2_gap = train_metrics["R2"] - test_metrics["R2"]
rmse_ratio = test_metrics["RMSE"] / train_metrics["RMSE"] if train_metrics["RMSE"] != 0 else np.inf

print(f"R2 gap (Train - Test): {r2_gap:.4f}")
print(f"RMSE ratio (Test / Train): {rmse_ratio:.4f}")

if r2_gap > 0.10 or rmse_ratio > 1.20:
    fit_status = "Potential overfitting"
else:
    fit_status = "Model appears reasonably generalized"

print(f"Model diagnosis: {fit_status}")

R2 gap (Train - Test): -0.0001
RMSE ratio (Test / Train): 0.9990
Model diagnosis: Model appears reasonably generalized


In [12]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
cv_rmse = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")

print(f"CV R2 scores: {np.round(cv_r2, 4)}")
print(f"Mean CV R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")
print(f"Mean CV RMSE: {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f}")

CV R2 scores: [0.9933 0.9933 0.9932 0.9931 0.9931]
Mean CV R2: 0.9932 +/- 0.0001
Mean CV RMSE: 2.5633 +/- 0.0171


## Interpretation and Overfitting Reduction Suggestions

Possible ways to reduce overfitting:
1. Feature selection: keep only meaningful variables and remove weak/noisy predictors.
2. Cross-validation: tune and validate model stability across folds.
3. Remove irrelevant variables: especially IDs, leakage-like fields, or redundant columns.
4. Regularization alternatives: try Ridge/Lasso/ElasticNet to shrink unstable coefficients.